# SpaceX Falcon 9 - Data Collection API

In this notebook, we collect SpaceX Falcon 9 launch data using the SpaceX REST API.

## Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

: 

## Helper functions
These functions use the launch data to extract booster, launch site, payload, and core (landing outcome) information from separate API endpoints, since these come as IDs in the main launch data.

In [ ]:
BoosterVersion = []

def getBoosterVersion(data):
    for x in data["rocket"]:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
            BoosterVersion.append(response["name"])

In [ ]:
LaunchSite = []
Longitude = []
Latitude = []

def getLaunchSite(data):
    for x in data["launchpad"]:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
            Longitude.append(response["longitude"])
            Latitude.append(response["latitude"])
            LaunchSite.append(response["name"])

In [ ]:
PayloadMass = []
Orbit = []

def getPayloadData(data):
    for load in data["payloads"]:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
            PayloadMass.append(response["mass_kg"])
            Orbit.append(response["orbit"])

In [ ]:
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []

def getCoreData(data):
    for core in data["cores"]:
        if core["core"] != None:
            response = requests.get("https://api.spacexdata.com/v4/cores/"+core["core"]).json()
            Block.append(response["block"])
            ReusedCount.append(response["reuse_count"])
            Serial.append(response["serial"])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core["landing_success"])+" "+str(core["landing_type"]))
        Flights.append(core["flight"])
        GridFins.append(core["gridfins"])
        Reused.append(core["reused"])
        Legs.append(core["legs"])
        LandingPad.append(core["landpad"])

## Request rocket launch data from SpaceX API

In [ ]:
spacex_url="https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)

Use the static response saved during course development (used if the live API changes over time), then normalize the JSON into a dataframe.

In [ ]:
static_json_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
response = requests.get(static_json_url)

data = pd.json_normalize(response.json())
data.head()

## Data Wrangling / Cleanup
We only need a subset of the columns, and we need to remove rows with multiple cores or payloads (those were not standard flights).

In [ ]:
data = data[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]]

data = data[data["cores"].map(len) == 1]
data = data[data["payloads"].map(len) == 1]

data["cores"] = data["cores"].map(lambda x: x[0])
data["payloads"] = data["payloads"].map(lambda x: x[0])

data["date"] = pd.to_datetime(data["date_utc"]).dt.date

data = data[data["date"] <= datetime.date(2020, 11, 13)]

## Extract additional data using the helper functions

In [ ]:
BoosterVersion = []
getBoosterVersion(data)

LaunchSite = []
Longitude = []
Latitude = []
getLaunchSite(data)

PayloadMass = []
Orbit = []
getPayloadData(data)

Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
getCoreData(data)

## Combine the collected data into a dictionary, then a dataframe

In [ ]:
launch_dict = {"FlightNumber": list(data["flight_number"]),
"Date": list(data["date"]),
"BoosterVersion": BoosterVersion,
"PayloadMass": PayloadMass,
"Orbit": Orbit,
"LaunchSite": LaunchSite,
"Outcome": Outcome,
"Flights": Flights,
"GridFins": GridFins,
"Reused": Reused,
"Legs": Legs,
"LandingPad": LandingPad,
"Block": Block,
"ReusedCount": ReusedCount,
"Serial": Serial,
"Longitude": Longitude,
"Latitude": Latitude}

df = pd.DataFrame(launch_dict)
df.head()

## Filter the dataframe to only Falcon 9 launches
Falcon 1 launches should be removed, since this project is specifically about predicting Falcon 9 first-stage landings.

In [ ]:
data_falcon9 = df[df["BoosterVersion"] != "Falcon 1"]
data_falcon9.reset_index(drop=True, inplace=True)
data_falcon9.loc[:, "FlightNumber"] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9.head()

## Data Wrangling: Handle Missing Values
Check for missing values, then replace missing PayloadMass values with the column mean (LandingPad missing values are left as-is, since they represent legitimate cases where no ground pad was used).

In [ ]:
data_falcon9.isnull().sum()

In [ ]:
payload_mean = data_falcon9["PayloadMass"].mean()
data_falcon9["PayloadMass"].replace(np.nan, payload_mean, inplace=True)

data_falcon9.isnull().sum()

## Export the cleaned dataset
This CSV will be used as the input for the next notebook (Data Wrangling).

In [ ]:
data_falcon9.to_csv("dataset_part_1.csv", index=False)